# GRPO Regime-Aware Factor Training v6.16
## Adaptive Exploration + Closed-Loop Control (Plan F)


In [ ]:
# -*- coding: utf-8 -*-"""GRPO Regime-Aware Factor Training v6.19 — Composite Score Full Fix + 8 Audit Repairsv6.16 關鍵新增 (基於 v6.15 Plan E):1. **【P0 Fix】Composite Score Best-Formula** — best_formula 改用 train_IC*0.3 + val_IC*0.7，打破 operator_bonus 虛高天花板2. **【P0 Fix】operator_bonus 2.0→0.5** — 減少虛高分數形成3. **【P0 Fix】best_val_ic 每步獨立追蹤** — 不限 early stop warmup，修復 -inf bug4. **【P1 Fix】移除 pip install torch** — 避免 cu118 安裝失敗浪費時間5. **【P1 Fix】CPU 單 regime mid_cap_tech** — 減少 CPU 訓練時間6. **【P1 Fix】CPU group_size=32** — 加速 CPU 訓練7. **【P2 Fix】強制重種降溫** — best_composite 連續 1000 步未更新時重置8. **【P2 Fix】early_stop_warmup 3000→1000** — 更早啟用 val_ic 追蹤9. 保留 v6.2~v6.17 全部功能"""import json, os, sys, time, mathimport numpy as npimport pandas as pdfrom typing import List, Dict, Tuple, Optionalfrom dataclasses import dataclass, fieldfrom enum import Enumfrom collections import defaultdict# ============================================================# 0. 環境檢查# ============================================================def check_environment():    print("=" * 60)    print(" 台股 GRPO Regime-Aware 因子訓練 (Kaggle GPU) - v6.19 - Composite Score Full Fix")    print("=" * 60)        import torch    gpu_compatible = False        # 【v6.16】GPU: 預期 T4 (sm_75) 相容 PyTorch 2.10+cu128    # 【v6.18】若 CUDA probe FAIL，直接 CPU 模式（不安裝 cu118）    if torch.cuda.is_available():        gpu_name = torch.cuda.get_device_name(0)        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9        cc = torch.cuda.get_device_capability(0)        print(f"  GPU: {gpu_name} ({gpu_mem:.1f} GB), sm_{cc[0]}{cc[1]}")        print(f"  PyTorch version: {torch.__version__}")                # CUDA probe 實測：唯一可靠的 GPU 可用性檢測        try:            _test = torch.zeros(1, device="cuda")            _ = _test + 1.0            _ = _test @ _test            torch.cuda.synchronize()            gpu_compatible = True            print(f"  [v6.15] CUDA probe: PASS → GPU 訓練")        except RuntimeError as e:            # 【v6.18】不再嘗試安裝 cu118 — P100 sm_60 已無可用 wheel            print(f"  [v6.18] CUDA probe FAIL — 直接使用 CPU 模式 (略過 cu118 安裝)")    else:        print("  [v6.15] No GPU detected → CPU fallback")        if not gpu_compatible:        os.environ["GRPO_FORCE_CPU"] = "1"        print("  >>> 使用 CPU 模式 (GRPO_FORCE_CPU=1)")        print(f"  PyTorch: {torch.__version__}")    print(f"  NumPy: {np.__version__}")    print(f"  Pandas: {pd.__version__}\n")# ============================================================# 1. 詞彙表與常數# ============================================================FEATURE_NAMES = (    "RET", "LIQ_SCORE", "PRESSURE", "FOMO", "DEV", "LOG_VOL",    "INST_FLOW", "MARGIN_PRESS", "FIVE_DAY_HIGH", "VOL_BREAKOUT",    "CVD_PROXY", "ABSORPTION", "SURF_ENTRY", "ATR", "CLOSE_POS", "MOM_REV",    "TX_INST_NET_OI", "MTX_RETAIL_OI", "TX_MTX_SPREAD",    "NASDAQ_CLOSE", "SP500_CLOSE", "DOWJONES_CLOSE",)OPERATOR_NAMES = (    "ADD", "SUB", "MUL", "DIV", "NEG", "ABS",    "SIGN", "GATE", "JUMP", "DECAY", "DELAY1", "MAX3",)OPERATOR_ARITY = [2, 2, 2, 2, 1, 1, 1, 3, 1, 1, 1, 1]N_FEATURES = len(FEATURE_NAMES)N_OPERATORS = len(OPERATOR_NAMES)VOCAB_SIZE = N_FEATURES + N_OPERATORS# ============================================================# 2. StackVM & StackVMState# ============================================================class StackVM:    def execute(self, tokens: List[int], feat_tensor: np.ndarray) -> Optional[np.ndarray]:        n_features = N_FEATURES        stack = []        for t in tokens:            if t < n_features:                stack.append(feat_tensor[t].copy())            else:                op_idx = t - n_features                if op_idx >= N_OPERATORS:                    return None                arity = OPERATOR_ARITY[op_idx]                if len(stack) < arity:                    return None                if arity == 1:                    a = stack.pop()                    stack.append(self._apply_unary(op_idx, a))                elif arity == 2:                    b, a = stack.pop(), stack.pop()                    stack.append(self._apply_binary(op_idx, a, b))                elif arity == 3:                    c, b, a = stack.pop(), stack.pop(), stack.pop()                    stack.append(self._apply_ternary(op_idx, a, b, c))        return stack[0] if len(stack) == 1 else None    @staticmethod    def _apply_unary(op_idx: int, a: np.ndarray):        if op_idx == 4:            return -a        if op_idx == 5:            return np.abs(a)        if op_idx == 6:            return np.sign(a)        if op_idx == 8:            return np.where(np.abs((a - np.mean(a)) / (np.std(a) + 1e-6)) > 3, np.sign(a), 0)        if op_idx == 9:            return 0.8 * a + 0.6 * np.roll(a, 1)        if op_idx == 10:            return np.roll(a, 1)        if op_idx == 11:            return np.maximum(np.maximum(a, np.roll(a, 1)), np.roll(a, 2))        return a    @staticmethod    def _apply_binary(op_idx: int, a: np.ndarray, b: np.ndarray):        if op_idx == 0:            return a + b        if op_idx == 1:            return a - b        if op_idx == 2:            return a * b        if op_idx == 3:            return a / (b + 1e-6)        return a    @staticmethod    def _apply_ternary(op_idx: int, a, b, c):        if op_idx == 7:            return np.where(c > 0, a, b)        return aclass StackVMState:    def __init__(self, max_stack_depth: int = 3):        self.stack_depth = 0        self.max_stack = max_stack_depth        def reset(self):        self.stack_depth = 0        def get_valid_tokens(self, position: int, remaining: int) -> set:        valid = set()        if self.stack_depth < self.max_stack:            new_depth = self.stack_depth + 1            min_needed = new_depth - 1            if remaining - 1 >= min_needed:                valid.update(range(N_FEATURES))        for op_idx in range(N_OPERATORS):            arity = OPERATOR_ARITY[op_idx]            if self.stack_depth >= arity:                new_depth = self.stack_depth - arity + 1                min_needed = new_depth - 1                if remaining - 1 >= min_needed:                    valid.add(N_FEATURES + op_idx)        return valid        def apply_token(self, token: int):        if token < N_FEATURES:            self.stack_depth += 1        else:            op_idx = token - N_FEATURES            arity = OPERATOR_ARITY[op_idx]            self.stack_depth = self.stack_depth - arity + 1        def is_complete(self) -> bool:        return self.stack_depth == 1# ============================================================# 3. Regime 定義 — 20 檔股票 (5/regime)# ============================================================class StockRegime(Enum):    LARGE_CAP = "large_cap"    MID_CAP_TECH = "mid_cap_tech"    TRADITIONAL = "traditional"    FINANCIAL = "financial"# [v6.11 修復] stock_id 改為 int 鍵，避免與 pandas groupby 型別不匹配KNOWN_REGIMES = {    # LARGE_CAP    2330: StockRegime.LARGE_CAP,   # 台積電    2308: StockRegime.LARGE_CAP,   # 大聯大    2412: StockRegime.LARGE_CAP,   # 中華電    1303: StockRegime.LARGE_CAP,   # 南亞    1326: StockRegime.LARGE_CAP,   # 台化    # MID_CAP_TECH    2454: StockRegime.MID_CAP_TECH,  # 聯發科    2382: StockRegime.MID_CAP_TECH,  # 廣達    2317: StockRegime.MID_CAP_TECH,  # 鴻海    3034: StockRegime.MID_CAP_TECH,  # 聯詠    3711: StockRegime.MID_CAP_TECH,  # 日月光投控    # TRADITIONAL    1301: StockRegime.TRADITIONAL,  # 台塑    1101: StockRegime.TRADITIONAL,  # 台泥    2002: StockRegime.TRADITIONAL,  # 中鋼    2105: StockRegime.TRADITIONAL,  # 正新    2207: StockRegime.TRADITIONAL,  # 和泰車    # FINANCIAL    2882: StockRegime.FINANCIAL,  # 國泰金    2886: StockRegime.FINANCIAL,  # 兆豐金    2891: StockRegime.FINANCIAL,  # 中信金    2881: StockRegime.FINANCIAL,  # 富邦金    2884: StockRegime.FINANCIAL,  # 玉山金}@dataclassclass RegimeConfig:    feature_weights: Dict[StockRegime, Dict[str, float]] = field(default_factory=lambda: {        StockRegime.LARGE_CAP: {"RET": 1.0, "LIQ_SCORE": 1.0, "PRESSURE": 1.5, "INST_FLOW": 2.0},        StockRegime.MID_CAP_TECH: {"RET": 1.0, "FOMO": 1.2, "FIVE_DAY_HIGH": 1.5, "VOL_BREAKOUT": 1.5},        StockRegime.TRADITIONAL: {"RET": 1.0, "DEV": 1.5, "VOL_BREAKOUT": 1.5, "MOM_REV": 1.0},        StockRegime.FINANCIAL: {"RET": 1.0, "LIQ_SCORE": 1.5, "CLOSE_POS": 1.5, "ATR": 1.2},    })    operator_mask: Dict[StockRegime, Dict[str, bool]] = field(default_factory=lambda: {        StockRegime.LARGE_CAP: {n: True for n in OPERATOR_NAMES},        StockRegime.MID_CAP_TECH: {n: True for n in OPERATOR_NAMES},        StockRegime.TRADITIONAL: {**{n: True for n in OPERATOR_NAMES}, "SIGN": False, "JUMP": False},        StockRegime.FINANCIAL: {**{n: True for n in OPERATOR_NAMES}, "SIGN": False, "JUMP": False},    })    training_params: Dict[StockRegime, Dict] = field(default_factory=lambda: {        # v6.13: group_size floor — TRADITIONAL/FINANCIAL 16→32        StockRegime.LARGE_CAP: {"group_size": 64, "reward_horizon": 5},        StockRegime.MID_CAP_TECH: {"group_size": 24, "reward_horizon": 4},        StockRegime.TRADITIONAL: {"group_size": 32, "reward_horizon": 5},        StockRegime.FINANCIAL: {"group_size": 32, "reward_horizon": 7},    })class RegimeTrainingPlan:    def __init__(self, config: RegimeConfig = None):        self.config = config or RegimeConfig()        def create_plan(self, stock_id: str, regime: StockRegime) -> dict:        weights_dict = self.config.feature_weights[regime]        feature_weights = np.array([weights_dict.get(f, 1.0) for f in FEATURE_NAMES], dtype=np.float32)        feature_mask = feature_weights > 0.8        ops_dict = self.config.operator_mask[regime]        operator_mask = np.array([ops_dict.get(op, True) for op in OPERATOR_NAMES], dtype=bool)        params = self.config.training_params[regime]        return {            "regime": regime,            "feature_weights": feature_weights,            "feature_mask": feature_mask,            "operator_mask": operator_mask,            "group_size": params["group_size"],            "reward_horizon": params["reward_horizon"],        }# ============================================================# 4. 特徵工程與 Robust Normalize# ============================================================def robust_normalize(arr, window=60):    if arr is None or len(arr) < window:        return arr    result = np.copy(arr).astype(np.float64)    for i in range(window, len(arr)):        segment = arr[i - window:i]        med = np.median(segment)        mad = np.median(np.abs(segment - med))        if mad > 1e-8:            result[i] = (arr[i] - med) / (1.4826 * mad)        else:            result[i] = 0.0    global_med = np.median(arr[:window])    global_mad = np.median(np.abs(arr[:window] - global_med))    if global_mad > 1e-8:        result[:window] = (arr[:window] - global_med) / (1.4826 * global_mad)    else:        result[:window] = 0.0    return resultclass TWFeatureEngineer:    NORM_WINDOW = 60    NORM_CLIP = 5.0    @staticmethod    def compute_features(df, inst_df=None, margin_df=None, futures_oi_df=None, us_indices_df=None):        result_frames = []        for stock_id, group in df.groupby("stock_id"):            g = group.sort_values("date").copy().reset_index(drop=True)            g["date"] = pd.to_datetime(g["date"])            g["ret"] = np.log(g["close"] / g["close"].shift(1))            g["liq_score"] = g["volume"] / (g["volume"].rolling(20).mean() + 1e-6)            g["pressure"] = np.tanh(3.0 * (g["close"] - g["open"]) / (g["high"] - g["low"] + 1e-6))            vol_chg = g["volume"].pct_change()            g["fomo"] = vol_chg - vol_chg.shift(1)            ma20 = g["close"].rolling(20).mean()            g["dev"] = (g["close"] - ma20) / (ma20 + 1e-6)            g["log_vol"] = np.log1p(g["volume"])            # 法人買賣超 — 從 inst_df merge 真實資料            if inst_df is not None and len(inst_df) > 0:                _inst_merge = inst_df[inst_df["stock_id"] == stock_id][["date", "total_net"]].copy()                if len(_inst_merge) > 0:                    _inst_merge["date"] = pd.to_datetime(_inst_merge["date"])                    _inst_merge = _inst_merge.rename(columns={"total_net": "inst_flow_raw"})                    g = g.merge(_inst_merge[["date", "inst_flow_raw"]], on="date", how="left")                    g["inst_flow"] = g["inst_flow_raw"].fillna(0)                    g.drop(columns=["inst_flow_raw"], inplace=True)                else:                    g["inst_flow"] = 0.0            else:                g["inst_flow"] = 0.0            # 融資融券壓力 — 從 margin_df merge 真實資料            if margin_df is not None and len(margin_df) > 0:                _mg_merge = margin_df[margin_df["stock_id"] == stock_id][["date", "margin_balance", "margin_change", "short_balance"]].copy()                if len(_mg_merge) > 0:                    _mg_merge["date"] = pd.to_datetime(_mg_merge["date"])                    _mg_merge["margin_press_raw"] = _mg_merge["margin_change"].fillna(0) / (_mg_merge["margin_balance"].fillna(1) + 1e-6)                    g = g.merge(_mg_merge[["date", "margin_press_raw"]], on="date", how="left")                    g["margin_press"] = g["margin_press_raw"].fillna(0)                    g.drop(columns=["margin_press_raw"], inplace=True)                else:                    g["margin_press"] = 0.0            else:                g["margin_press"] = 0.0            high5 = g["close"].rolling(5).max()            g["five_day_high"] = (g["close"] - high5) / (high5 + 1e-6)            vol_ma5 = g["volume"].rolling(5).mean()            g["vol_breakout"] = g["volume"] / (vol_ma5 + 1e-6)            cvd_intraday = (g["close"] - g["open"]) / (g["high"] - g["low"] + 1e-6) * g["volume"]            g["cvd_proxy"] = cvd_intraday.rolling(20).sum() / (g["volume"].rolling(20).mean() * 20 + 1e-6)            g["absorption"] = (g["high"] - g["close"]) / (g["high"] - g["low"] + 1e-6) * g["volume"]                        # v6.0 dsfix: 從 futures_oi_df 和 us_indices_df merge 真實資料            # 期貨 OI: pivot TX/MTX wide 再 merge (避免每檔股票 rows 翻倍)            if futures_oi_df is not None and len(futures_oi_df) > 0:                foi = futures_oi_df.copy()                foi["date"] = pd.to_datetime(foi["date"])                tx_oi = foi[foi["futures_id"] == "TX"][["date", "inst_net_oi", "retail_net_oi"]].copy()                tx_oi = tx_oi.rename(columns={"inst_net_oi": "tx_inst_net_oi", "retail_net_oi": "tx_retail_net_oi"})                mtx_oi = foi[foi["futures_id"] == "MTX"][["date", "inst_net_oi", "retail_net_oi"]].copy()                mtx_oi = mtx_oi.rename(columns={"inst_net_oi": "mtx_inst_net_oi", "retail_net_oi": "mtx_retail_net_oi"})                g = g.merge(tx_oi, on="date", how="left")                g = g.merge(mtx_oi, on="date", how="left")                g["TX_INST_NET_OI"] = g["tx_inst_net_oi"].fillna(0)                g["MTX_RETAIL_OI"] = g["mtx_retail_net_oi"].fillna(0)                # TX_MTX_SPREAD                g["TX_MTX_SPREAD"] = (g["tx_inst_net_oi"].fillna(0) - g["mtx_inst_net_oi"].fillna(0))                # 清理 merge 殘留欄位                for c in ["tx_inst_net_oi", "tx_retail_net_oi", "mtx_inst_net_oi", "mtx_retail_net_oi"]:                    if c in g.columns: g.drop(columns=[c], inplace=True)            else:                for f in ["TX_INST_NET_OI", "MTX_RETAIL_OI", "TX_MTX_SPREAD"]:                    g[f] = 0.0                        # 美股指數: pivot wide per index 再 merge            if us_indices_df is not None and len(us_indices_df) > 0:                us = us_indices_df.copy()                us["date"] = pd.to_datetime(us["date"])                for idx_name, feat_name in [("Nasdaq", "NASDAQ_CLOSE"),                                             ("SP500", "SP500_CLOSE"),                                             ("DowJones", "DOWJONES_CLOSE")]:                    idx_data = us[us["index_name"] == idx_name][["date", "close"]].copy()                    idx_data = idx_data.rename(columns={"close": feat_name})                    g = g.merge(idx_data, on="date", how="left")                    g[feat_name] = g[feat_name].fillna(0).shift(1).fillna(0)  # 美股時差            else:                for f in ["NASDAQ_CLOSE", "SP500_CLOSE", "DOWJONES_CLOSE"]:                    g[f] = 0.0                        key_level = g["close"].rolling(20).mean()            g["surf_entry"] = np.where(                np.abs(g["close"] - key_level) / (key_level + 1e-6) < 0.01, 1.0, 0.0            )            high, low, close = g["high"].values, g["low"].values, g["close"].values            prev_close = np.roll(close, 1)            prev_close[0] = close[0]            tr = np.maximum(high - low, np.maximum(np.abs(high - prev_close), np.abs(low - prev_close)))            g["atr"] = pd.Series(tr).rolling(14).mean() / (close + 1e-6)            g["close_pos"] = (g["close"] - g["low"]) / (g["high"] - g["low"] + 1e-6)            g["mom_rev"] = -1 * g["ret"].rolling(5).sum()            _lower_to_upper = {                "ret": "RET", "liq_score": "LIQ_SCORE", "pressure": "PRESSURE",                "fomo": "FOMO", "dev": "DEV", "log_vol": "LOG_VOL",                "inst_flow": "INST_FLOW", "margin_press": "MARGIN_PRESS",                "five_day_high": "FIVE_DAY_HIGH", "vol_breakout": "VOL_BREAKOUT",                "cvd_proxy": "CVD_PROXY", "absorption": "ABSORPTION",                "surf_entry": "SURF_ENTRY", "atr": "ATR",                "close_pos": "CLOSE_POS", "mom_rev": "MOM_REV",            }            for _lower, _upper in _lower_to_upper.items():                if _lower in g.columns:                    g[_upper] = g[_lower]                        for feat in FEATURE_NAMES:                if feat not in g.columns:                    g[feat] = 0.0                    continue                g[feat] = robust_normalize(g[feat].values, window=TWFeatureEngineer.NORM_WINDOW)                g[feat] = g[feat].clip(-TWFeatureEngineer.NORM_CLIP, TWFeatureEngineer.NORM_CLIP)                        keep_cols = ["date", "stock_id", "close"] + list(FEATURE_NAMES)            result_frames.append(g[keep_cols].copy())                return pd.concat(result_frames, ignore_index=True)# ============================================================# 5. 【v5.9 關鍵修復】合成數據生成器 — 橫斷面差異# ============================================================def generate_synthetic_data(n_days: int = 750, seed: int = 42):    """【已禁用】合成數據 — 充滿噪聲，無效訓練。    v6.1+ 強制使用真實 Kaggle dataset，此函數不應被呼叫。    """    raise RuntimeError(        "[FATAL] 合成數據已禁用！請確認 Kaggle dataset 掛載正確。"        " 預期: /kaggle/input/twstock-v6-0-real-data-20stocks-5y/ 包含 price_ohlcv.csv 等真實資料。"    )# ============================================================# 5b. 真實數據載入 (Kaggle dataset)# ============================================================def adapt_finmind_data(data_path: str):    """從 Kaggle dataset 載入真實台股數據 — 自動掃描 /kaggle/input/ 下所有 CSV"""    import os, glob    # 自動掃描 /kaggle/input/ 下所有 CSV    csv_candidates = []    kaggle_input = "/kaggle/input"    if os.path.exists(kaggle_input):        for root, dirs, files in os.walk(kaggle_input):            for f in files:                if f.endswith('.csv'):                    csv_candidates.append(os.path.join(root, f))        if len(csv_candidates) == 0:        # 也試試 data_path 本身        if os.path.exists(data_path):            for root, dirs, files in os.walk(data_path):                for f in files:                    if f.endswith('.csv'):                        csv_candidates.append(os.path.join(root, f))        csv_files = {}    for cp in csv_candidates:        fname = os.path.basename(cp)        csv_files[fname] = cp    print(f"  [adapt_finmind] 找到 {len(csv_files)} 個 CSV: {list(csv_files.keys())}")    if len(csv_files) == 0:        # Diagnostic: list /kaggle/input/        print("  [adapt_finmind] /kaggle/input 內容:")        if os.path.exists(kaggle_input):            for root, dirs, files in os.walk(kaggle_input):                for d in dirs:                    print(f"    {d}/")                for f in files:                    print(f"      {f}")        print("  [adapt_finmind] 找不到任何 CSV 檔案，返回 None")        return None, None, None, None, None    # 1. OHLCV — 多檔名掃描    ohlcv_candidates = ["twstock_daily.csv", "price_ohlcv.csv", "ohlcv.csv"]    ohlcv_path = None    for cand in ohlcv_candidates:        if cand in csv_files:            ohlcv_path = csv_files[cand]            break    if ohlcv_path is None:        print("  [adapt_finmind] 找不到 OHLCV CSV, 返回 None")        return None, None, None, None, None    df = pd.read_csv(ohlcv_path)    col_map = {"\u4ee3\u865f": "stock_id", "\u540d\u7a31": "name", "\u65e5\u671f": "date",               "\u958b\u76e4\u50f9": "open", "\u6700\u9ad8\u50f9": "high", "\u6700\u4f4e\u50f9": "low",               "\u6536\u76e4\u50f9": "close", "\u6210\u4ea4\u80a1\u6578": "volume"}    df.rename(columns={k: v for k, v in col_map.items() if k in df.columns}, inplace=True)    for col in ["open", "high", "low", "close", "volume"]:        if col in df.columns:            df[col] = pd.to_numeric(df[col], errors="coerce")    df.dropna(subset=["close"], inplace=True)    if "date" in df.columns:        df["date"] = pd.to_datetime(df["date"])    if "stock_id" not in df.columns and "\u4ee3\u865f" in ohlcv_path:        pass    print(f"  [adapt_finmind] \u8f09\u5165 OHLCV: {len(df)} rows, {df['stock_id'].nunique() if 'stock_id' in df.columns else '?'} stocks")    # 2. 法人買賣超    inst_df = None    inst_candidates = ["inst_flow.csv", "inst_data.csv"]    inst_path = None    for cand in inst_candidates:        if cand in csv_files:            inst_path = csv_files[cand]            break    if inst_path is not None:        raw = pd.read_csv(inst_path)        if "date" in raw.columns:            raw["date"] = pd.to_datetime(raw["date"])        investor_cols = [c for c in ["Foreign_Investor", "Dealer_self", "Investment_Trust",                                       "Dealer_Hedging", "Foreign_Dealer_Self"]                         if c in raw.columns]        if investor_cols and "name" not in raw.columns:            name_map = {                "Foreign_Investor": "foreign_net",                "Foreign_Dealer_Self": "foreign_dealer_net",                "Investment_Trust": "trust_net",                "Dealer_self": "dealer_self_net",                "Dealer_Hedging": "dealer_hedging_net",            }            raw.rename(columns=name_map, inplace=True)            inst_cols = [c for c in ["foreign_net", "trust_net", "dealer_self_net"]                         if c in raw.columns]            if inst_cols:                raw["total_net"] = raw[inst_cols].sum(axis=1)            else:                raw["total_net"] = 0            inst_df = raw            print(f"  [adapt_finmind] inst: pre-mapped ({len(inst_df)} rows)")        elif "name" in raw.columns:            pivot = raw.pivot_table(                index=["date", "stock_id"], columns="name",                values="net", aggfunc="sum"            ).reset_index()            name_map = {                "Foreign_Investor": "foreign_net",                "Foreign_Dealer_Self": "foreign_dealer_net",                "Investment_Trust": "trust_net",                "Dealer_self": "dealer_self_net",                "Dealer_Hedging": "dealer_hedging_net",            }            pivot.columns = [name_map.get(c, c) for c in pivot.columns]            inst_cols = [c for c in ["foreign_net", "trust_net", "dealer_self_net"]                         if c in pivot.columns]            if inst_cols:                pivot["total_net"] = pivot[inst_cols].sum(axis=1)            else:                pivot["total_net"] = 0            inst_df = pivot            print(f"  [adapt_finmind] inst: long format pivot ({len(inst_df)} rows)")        else:            inst_df = raw            print(f"  [adapt_finmind] inst: direct ({len(inst_df)} rows)")    else:        print("  [adapt_finmind] inst: \u627e\u4e0d\u5230\u6cd5\u4eba\u8cc7\u6599")    # 3. \u878d\u8cc7\u878d\u5238    margin_df = None    margin_candidates = ["margin.csv", "margin_data.csv"]    margin_path = None    for cand in margin_candidates:        if cand in csv_files:            margin_path = csv_files[cand]            break    if margin_path is not None:        raw = pd.read_csv(margin_path)        if "date" in raw.columns:            raw["date"] = pd.to_datetime(raw["date"])        if "margin_balance" in raw.columns:            margin_df = raw            print(f"  [adapt_finmind] margin: pre-mapped ({len(margin_df)} rows)")        elif "MarginPurchaseTodayBalance" in raw.columns:            if "MarginPurchaseTodayBalance" in raw.columns:                raw["margin_balance"] = raw["MarginPurchaseTodayBalance"]            if "MarginPurchaseBuy" in raw.columns:                raw["margin_buy"] = raw["MarginPurchaseBuy"]            if "ShortSaleTodayBalance" in raw.columns:                raw["short_balance"] = raw["ShortSaleTodayBalance"]            margin_df = raw            print(f"  [adapt_finmind] margin: FinMind ({len(margin_df)} rows)")        else:            margin_df = raw            print(f"  [adapt_finmind] margin: direct ({len(margin_df)} rows)")    else:        print("  [adapt_finmind] margin: \u627e\u4e0d\u5230\u878d\u8cc7\u8cc7\u6599")    # 4. \u671f\u8ca8 OI    futures_oi_df = None    foi_candidates = ["futures_oi.csv"]    foi_path = None    for cand in foi_candidates:        if cand in csv_files:            foi_path = csv_files[cand]            break    if foi_path is not None:        raw = pd.read_csv(foi_path)        if "date" in raw.columns:            raw["date"] = pd.to_datetime(raw["date"])        if "futures_id" in raw.columns and "inst_net_oi" in raw.columns and len(raw) > 0:            cols = [c for c in ["date", "futures_id", "inst_net_oi", "retail_net_oi"] if c in raw.columns]            futures_oi_df = raw[cols].copy()            if "retail_net_oi" not in raw.columns:                if "total_oi" in raw.columns:                    futures_oi_df["retail_net_oi"] = raw["total_oi"] - raw["inst_net_oi"]                else:                    futures_oi_df["retail_net_oi"] = 0            futures_oi_df = futures_oi_df.sort_values(["date", "futures_id"]).reset_index(drop=True)            print(f"  [adapt_finmind] futures: long format ({len(futures_oi_df)} rows, {futures_oi_df['futures_id'].nunique()} contracts)")        elif "oi" in raw.columns and len(raw) > 0:            near_month = raw.loc[raw.groupby(["date"])["oi"].idxmax()]            near_month = near_month[["date", "oi"]].copy()            near_month = near_month.rename(columns={"oi": "total_oi"})            near_month["inst_net_oi"] = near_month["total_oi"] * 0.6            near_month["retail_net_oi"] = near_month["total_oi"] * 0.4            near_month["futures_id"] = "TX"            if "mtx_oi" in raw.columns:                mtx_near = raw.loc[raw.groupby(["date"])["mtx_oi"].idxmax()]                mtx_rows = mtx_near[["date", "mtx_oi"]].copy()                mtx_rows = mtx_rows.rename(columns={"mtx_oi": "total_oi"})                mtx_rows["inst_net_oi"] = mtx_rows["total_oi"] * 0.6                mtx_rows["retail_net_oi"] = mtx_rows["total_oi"] * 0.4                mtx_rows["futures_id"] = "MTX"                futures_oi_df = pd.concat([near_month, mtx_rows], ignore_index=True)            else:                futures_oi_df = near_month            futures_oi_df = futures_oi_df.sort_values(["date", "futures_id"]).reset_index(drop=True)            print(f"  [adapt_finmind] futures: wide format (60/40 approx, {len(futures_oi_df)} rows)")        elif len(raw) > 0:            num_cols = raw.select_dtypes(include=[np.number]).columns.tolist()            num_cols = [c for c in num_cols if c != "date"]            if num_cols:                raw["inst_net_oi"] = raw[num_cols[0]] * 0.6                raw["retail_net_oi"] = raw[num_cols[0]] * 0.4                raw["futures_id"] = "TX"                cols = [c for c in ["date", "futures_id", "inst_net_oi", "retail_net_oi"] if c in raw.columns]                futures_oi_df = raw[cols].copy().sort_values("date").reset_index(drop=True)                print(f"  [adapt_finmind] futures: fallback ({len(futures_oi_df)} rows, using {num_cols[0]})")    else:        print("  [adapt_finmind] futures: \u627e\u4e0d\u5230\u671f\u8ca8\u8cc7\u6599")    # 5. \u7f8e\u80a1\u6307\u6578    us_indices_df = None    us_candidates = ["us_indices.csv"]    us_path = None    for cand in us_candidates:        if cand in csv_files:            us_path = csv_files[cand]            break    if us_path is not None:        raw = pd.read_csv(us_path)        if "date" in raw.columns:            raw["date"] = pd.to_datetime(raw["date"])        if "index_name" in raw.columns:            us_indices_df = raw            print(f"  [adapt_finmind] us: long format ({len(us_indices_df)} rows, {us_indices_df['index_name'].nunique()} indices)")        else:            records = []            for idx_name, col in [("Nasdaq", "NASDAQ"), ("SP500", "SP500"), ("DowJones", "DOWJONES")]:                if col in raw.columns:                    for _, row in raw.iterrows():                        if pd.notna(row.get(col)):                            records.append({"date": row["date"], "index_name": idx_name, "close": row[col]})            if records:                us_indices_df = pd.DataFrame(records)                print(f"  [adapt_finmind] us: wide→long ({len(us_indices_df)} rows)")    else:        print("  [adapt_finmind] us: \u627e\u4e0d\u5230\u7f8e\u80a1\u8cc7\u6599")    print(f"  [adapt_finmind] \u7d50\u679c: OHLCV={len(df)} rows, Inst={len(inst_df) if inst_df is not None else 0}, "          f"Margin={len(margin_df) if margin_df is not None else 0}, "          f"Futures={len(futures_oi_df) if futures_oi_df is not None else 0}, "          f"US={len(us_indices_df) if us_indices_df is not None else 0}")    return df, inst_df, margin_df, futures_oi_df, us_indices_df# ============================================================# 6. 【v5.9 穩定性優化】GRPOConfig# ============================================================@dataclassclass GRPOConfig:    group_size: int = 64          # v6.5: 從16提升到64，CPU模式需更大group防坍縮    d_model: int = 64    nhead: int = 4    num_layers: int = 2    dim_feedforward: int = 128    num_loops: int = 1    vocab_size: int = 34    max_formula_len: int = 15    batch_size: int = 256         # v6.5: 配合group_size=64 增加batch    train_steps: int = 12000      # v6.5: CPU環境妥協，從8000→15000 (原預設20000)    lr: float = 5e-4    entropy_coef: float = 0.25    # v6.5: 從0.08提升到0.15，強化探索    reward_clip: float = 5.0    advantage_clip: float = 3.0    # --- v6.12 新增/修改參數 ---    advantage_method: str = "rank"      # "rank" (v6.11) 或 "zscore" (v6.8 舊法)    min_group_size: int = 8            # 動態 group_size 下限    early_stop_patience: int = 3000      # 【v6.16】1200→2000    early_stop_min_delta: float = 5e-4  # 【v6.12】1e-4→5e-4，需更明顯改善才算    early_stop_warmup: int = 1000       # 【v6.18】3000→1000，更早啟用 val_ic 追蹤    # --- v6.19: Composite Score 參數 ---    composite_ic_weight: float = 0.3           # 【v6.19】train_IC 權重    composite_val_ic_weight: float = 0.7       # 【v6.19】val_IC 權重    composite_operator_tiny_bonus: float = 0.05 # 【v6.19】含 operator 的小獎勵    composite_bypass_min_len: bool = False     # 【v6.19】跳過 min_formula_len 檢查    # --- v6.19: Exploration stagnation recovery ---    exploration_stagnation_steps: int = 1000    exploration_stagnation_first_step: int = 500    exploration_stagnation_composite_threshold: float = 0.01    exploration_stagnation_eps_target: float = 0.3    exploration_stagnation_reseed_ratio: float = 0.7    min_formula_len: int = 4            # 【v6.16】3→4，強迫更長公式    operator_bonus: float = 0.5         # 【v6.18】2.0→0.5，調降避免虛高天花板    short_formula_penalty: float = 5.0  # 【v6.16】3.0→5.0，更強短公式懲罰    # --- v6.15 Plan E: Operator Exploration 參數 ---    operator_seed_ratio: float = 0.5     # 【v6.15A】warmup 中預填 operator 公式的比例    operator_epsilon_start: float = 0.3  # 【v6.15C】position 0 強制選 operator 的初始概率    operator_epsilon_decay: float = 0.9  # 【v6.15C】每 500 步 epsilon 衰減因子    feature_logit_bias_start: float = -2.0  # 【v6.15D】position 0 feature logit 壓制初始值    feature_logit_bias_decay_steps: int = 2000  # 【v6.15D】logit bias 線性歸零步數    # --- v6.16: 自適應探索控制參數 ---    adaptive_exploration: bool = True    exploration_recovery_ops_threshold: float = 0.3    exploration_recovery_len_threshold: float = 1.5    exploration_recovery_eps_boost: float = 0.8    exploration_recovery_fbias_boost: float = -2.0    exploration_recovery_reseed_ratio: float = 0.5    periodic_reseed_interval: int = 300    periodic_reseed_ratio: float = 0.3    adv_std_threshold: float = 0.05      # 低於此值觸發 group_size 縮小    reward_weights: dict = field(default_factory=lambda: {"ic": 0.30, "sharpe": 0.12, "mdd": 0.06, "turnover": 0.03, "complexity": 0.45, "simplicity": 0.08, "length": 0.10})  # 【v6.13】complexity 終於生效    use_multi_objective: bool = True     # True=多目標 reward (v6.12), False=IC only    use_overfit_penalty: bool = True    ic_gap_threshold: float = 0.05    ic_gap_weight: float = 5.0    use_val_ic_reward: bool = True    use_lord: bool = True    lord_decay: float = 1e-3    clip_eps: float = 0.2    guided_decoding: bool = True    warmup_steps: int = 500    max_stack_depth: int = 3    device: str = "cpu"    regimes_to_train: list = field(default_factory=list)  # 【v6.18】空=全部, CPU時=["mid_cap_tech"]    # v5.9 新增參數    gumbel_noise_scale: float = 2.5   # 【v6.16】1.5→2.5，更強探索噪聲    diversity_penalty: float = 3.0    # v6.5: 從2.0提升到3.0，加強多樣性懲罰    top_k: int = 0                    # 0=不限制, >0=只取top-k token    nucleus_threshold: float = 0.0    # 0=不限制, >0=nucleus sampling閾值    temperature_start: float = 2.0    # 初始溫度 (高=探索)    temperature_end: float = 1.0      # 【v6.16】0.8→1.0，保持更高探索溫度    temperature_decay_steps: int = 8000  # 溫度衰減步數    @classmethod    def auto_detect(cls):        config = cls()        try:            import torch            if torch.cuda.is_available() and os.environ.get("GRPO_FORCE_CPU", "0") != "1":                config.device = "cuda"                # 【v6.12】GPU 模式不設 group_size，讓 regime_plan 決定            else:                config.device = "cpu"                # 【v6.12】CPU 模式不強制覆蓋 group_size！                # group_size 由 train_torch_regime() 內的 regime_plan["group_size"] 決定                # 這裡只調整 train_steps 和 batch_size                config.train_steps = 6000   # 【v6.18】CPU 模式減少步數（單 regime）                config.batch_size = 128                config.group_size = 32       # 【v6.18】CPU 模式 group_size 64→32                config.regimes_to_train = ["mid_cap_tech"]  # 【v6.18】CPU 只訓練單 regime                print(f"[auto_detect v6.18] CPU mode - train_steps={config.train_steps}, batch_size={config.batch_size}, group_size={config.group_size}")                print(f"[auto_detect v6.18] CPU single regime: mid_cap_tech")        except ImportError:            pass        print(f"[DEBUG auto_detect] Final: group_size={config.group_size}, device={config.device}, GRPO_FORCE_CPU={os.environ.get('GRPO_FORCE_CPU', 'NOT_SET')}")        return configclass GitHubLogPusher:    def __init__(self, token=None, repo="milo0914/AlphaGPT", branch="main"):        self.enabled = False    def push_log(self, regime, step, metrics):        pass    def push_final(self, regime, result):        pass# ============================================================# 7. GRPO Reward Calculator# ============================================================class GRPORewardCalculator:    def __init__(self, config: GRPOConfig = None):        self.config = config or GRPOConfig()        self.vm = StackVM()    @staticmethod    def _spearman_corr(x, y):        rx = np.argsort(np.argsort(x)).astype(float)        ry = np.argsort(np.argsort(y)).astype(float)        rx = (rx - rx.mean()) / (rx.std() + 1e-8)        ry = (ry - ry.mean()) / (ry.std() + 1e-8)        return float(np.mean(rx * ry))    def _default_backtest(self, signal, returns):        """v6.13: 回傳 reward 分項 dict，由 compute_group_rewards 用 reward_weights 組合"""        valid = np.isfinite(signal) & np.isfinite(returns)        if valid.sum() < 10:            return {"ic": -1.0, "sharpe": 0.0, "mdd_penalty": 0.0, "turnover_penalty": 0.0, "_invalid": True}        sig = signal[valid]        ret = returns[valid]                # 1. IC (Spearman)        ic = self._spearman_corr(sig, ret)                if not self.config.use_multi_objective:            return {"ic": ic * 10, "sharpe": 0.0, "mdd_penalty": 0.0, "turnover_penalty": 0.0, "_invalid": False}                # 2. Sharpe Ratio (信號加權收益)        port_ret = sig * ret        sharpe = port_ret.mean() / (port_ret.std() + 1e-6) if port_ret.std() > 1e-6 else 0.0                # 3. MDD penalty (最大回撤)        cum_ret = np.cumprod(1 + port_ret)        running_max = np.maximum.accumulate(cum_ret)        drawdown = (cum_ret - running_max) / (running_max + 1e-8)        mdd = np.abs(drawdown.min()) if len(drawdown) > 0 else 0.0        mdd_penalty = -mdd * 2.0  # MDD 越大懲罰越重                # 4. Turnover penalty (信號變動率)        sig_changes = np.abs(np.diff(sig))        turnover = sig_changes.mean() / (np.abs(sig).mean() + 1e-6) if len(sig) > 1 else 0.0        turnover_penalty = -turnover * 0.5  # 高周轉率懲罰                # 【v6.13】回傳 dict — 不再自行組合！讓 compute_group_rewards 用 reward_weights 組合        # IC 縮放 10→5, Sharpe 縮放 5→2: 降低負 reward 幅度讓 complexity 能翻盤        return {            "ic": ic * 5,            # 【v6.13】10→5            "sharpe": sharpe * 2,    # 【v6.13】5→2            "mdd_penalty": mdd_penalty,            "turnover_penalty": turnover_penalty,            "_invalid": False,        }    def compute_group_rewards(self, group_tokens, feat_tensor, returns, train_ic=None, val_ic=None):        G = len(group_tokens)        rewards = []        reward_details = []  # 【v6.13】記錄每個 formula 的 reward 分項供 debug        if train_ic is None:            train_ic = np.zeros(G)        if val_ic is None:            val_ic = np.zeros(G)        for i, tokens in enumerate(group_tokens):            signal = self.vm.execute(tokens, feat_tensor)            if signal is None or np.std(signal) < 1e-4:                rewards.append(-5.0)                reward_details.append({"_invalid": True, "n_ops": 0, "f_len": len(tokens), "complexity_comp": 0.0})                continue            bt = self._default_backtest(signal, returns)  # 【v6.13】回傳 dict            if bt.get("_invalid"):                rewards.append(-5.0)                reward_details.append({"_invalid": True, "n_ops": 0, "f_len": len(tokens), "complexity_comp": 0.0})                continue                        # 【v6.13 根本修復】用 reward_weights 組合全部分項！            # v6.12 bug: _default_backtest 自行組合 ic/sharpe/mdd/turnover，            # reward_weights["complexity"]=0.25 從未被使用 → complexity 等於虛設            w = self.config.reward_weights            n_operators = sum(1 for t in tokens if t >= N_FEATURES)            f_len = len(tokens)                        # Base reward components (financial metrics)            combined = (w.get("ic", 0.35) * bt["ic"] +                        w.get("sharpe", 0.15) * bt["sharpe"] +                        w.get("mdd", 0.08) * bt["mdd_penalty"] +                        w.get("turnover", 0.04) * bt["turnover_penalty"])                        # 【v6.19】Val-IC 雙向獎懲 — 不只懲罰負值，還獎勵正值            if self.config.use_overfit_penalty and self.config.use_val_ic_reward:                t_ic, v_ic = train_ic[i], val_ic[i]                val_bonus = max(v_ic, 0.0) * 2.0      # 新增：正值獎勵                val_penalty = abs(v_ic) * 10.0 if v_ic < 0 else 0.0  # 保留                ic_gap = max(0, abs(t_ic) - abs(v_ic) - self.config.ic_gap_threshold)                ic_gap_penalty = self.config.ic_gap_weight * ic_gap                combined += val_bonus                combined -= (val_penalty + ic_gap_penalty)                        # 【v6.13】Complexity: 終於使用 reward_weights["complexity"]！            complexity_score = n_operators * self.config.operator_bonus            complexity_comp = w.get("complexity", 0.25) * complexity_score            combined += complexity_comp                        # 【v6.13】Simplicity: 短公式懲罰            simplicity_comp = 0.0            if f_len < self.config.min_formula_len:                simplicity_comp = w.get("simplicity", 0.05) * self.config.short_formula_penalty * (self.config.min_formula_len - f_len)                combined -= simplicity_comp                        # 【v6.13】Length: 避免極短公式            length_comp = w.get("length", 0.08) * max(0, f_len - 1) * 0.5            combined += length_comp                        detail = {                "ic_comp": round(w.get("ic", 0.35) * bt["ic"], 3),                "sharpe_comp": round(w.get("sharpe", 0.15) * bt["sharpe"], 3),                "mdd_comp": round(w.get("mdd", 0.08) * bt["mdd_penalty"], 3),                "turnover_comp": round(w.get("turnover", 0.04) * bt["turnover_penalty"], 3),                "complexity_comp": round(complexity_comp, 3),                "simplicity_comp": round(simplicity_comp, 3),                "length_comp": round(length_comp, 3),                "n_ops": n_operators,                "f_len": f_len,                "_invalid": False,            }            reward_details.append(detail)            rewards.append(np.clip(combined, -self.config.reward_clip, self.config.reward_clip))                rewards = np.array(rewards)        valid_mask = rewards > -5.0                # v6.11: Rank-Based Advantage — 永不 collapse        advantages = np.zeros_like(rewards)        n_valid = valid_mask.sum()        adv_std = 0.0        need_reduce_group = False                if n_valid > 1:            valid_rewards = rewards[valid_mask]            if self.config.advantage_method == "rank":                # === Rank-Based Advantage (v6.11) ===                # 排名歸一化：advantages 在 [-1, 1]，std≈0.577 永不 collapse                sorted_indices = np.argsort(np.argsort(valid_rewards))                ranks = sorted_indices.astype(float)                advantages_valid = (ranks - len(ranks) / 2) / (len(ranks) / 2)                advantages[valid_mask] = advantages_valid                advantages[~valid_mask] = -1.0                adv_std = advantages_valid.std()            else:                # === Z-score (v6.8 舊法，保留相容) ===                group_mean = valid_rewards.mean()                group_std = valid_rewards.std() + 1e-6                advantages_valid = (valid_rewards - group_mean) / group_std                advantages[valid_mask] = advantages_valid                advantages[~valid_mask] = 0.0                adv_std = advantages_valid.std()                        # v6.11: Dynamic Group Size — adv_std 太低時標記需縮小 group            if adv_std < self.config.adv_std_threshold:                need_reduce_group = True                print(f"  [v6.11] adv_std={adv_std:.4f} < {self.config.adv_std_threshold}, need_reduce_group=True")        elif n_valid == 1:            advantages[valid_mask] = 1.0            advantages[~valid_mask] = -1.0            adv_std = 0.0        else:            advantages[:] = -1.0            adv_std = 0.0        return {            "rewards": rewards,            "advantages": advantages,            "valid_mask": valid_mask,            "group_mean_reward": rewards[valid_mask].mean() if valid_mask.sum() > 0 else 0.0,            "best_idx": int(np.argmax(rewards)) if len(rewards) > 0 else 0,            "adv_std": float(adv_std),       # v6.11 新增            "need_reduce_group": need_reduce_group,  # v6.11 新增            "reward_details": reward_details,  # 【v6.13】        }# ============================================================# 8. LoopedTransformer 模型架構# ============================================================def _safe_logits(logits):    import torch    return torch.clamp(torch.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0), -20.0, 20.0)def build_looped_transformer(config):    import torch    import torch.nn as nn    import torch.nn.functional as F    class RMSNorm(nn.Module):        def __init__(self, d, eps=1e-6):            super().__init__()            self.w = nn.Parameter(torch.ones(d))            self.eps = eps        def forward(self, x):            return self.w * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))    class SwiGLU(nn.Module):        def __init__(self, d_model, d_ff, drop=0.1):            super().__init__()            self.w_gate = nn.Linear(d_model, d_ff, bias=False)            self.w_up = nn.Linear(d_model, d_ff, bias=False)            self.w_down = nn.Linear(d_ff, d_model, bias=False)            self.drop = nn.Dropout(drop)        def forward(self, x):            return self.drop(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))    class QKNormAttention(nn.Module):        def __init__(self, d_model, nhead, drop=0.1):            super().__init__()            self.d_k = d_model // nhead            self.nhead = nhead            self.w_q, self.w_k, self.w_v, self.w_o = [                nn.Linear(d_model, d_model, bias=False) for _ in range(4)            ]            self.q_norm, self.k_norm = RMSNorm(self.d_k), RMSNorm(self.d_k)            self.drop = nn.Dropout(drop)        def forward(self, x, mask=None):            B, T, D = x.shape            q = self.q_norm(self.w_q(x).view(B, T, self.nhead, self.d_k).transpose(1, 2))            k = self.k_norm(self.w_k(x).view(B, T, self.nhead, self.d_k).transpose(1, 2))            v = self.w_v(x).view(B, T, self.nhead, self.d_k).transpose(1, 2)            attn = F.softmax(torch.matmul(q, k.transpose(-2, -1)) * (self.d_k ** -0.5), dim=-1)            out = torch.matmul(self.drop(attn), v).transpose(1, 2).contiguous().view(B, T, D)            return self.w_o(out)    class MTPHead(nn.Module):        def __init__(self, d_model, vocab_size):            super().__init__()            self.h_mean, self.h_max, self.h_first = [                nn.Linear(d_model, vocab_size) for _ in range(3)            ]            self.gate = nn.Linear(d_model * 3, 3, bias=False)            self.critic = nn.Linear(d_model, 1)        def forward(self, h):            p_mean, p_max, p_first = h.mean(dim=1), h.max(dim=1).values, h[:, 0, :]            l_mean = self.h_mean(p_mean)            l_max = self.h_max(p_max)            l_first = self.h_first(p_first)            w = F.softmax(self.gate(torch.cat([p_mean, p_max, p_first], dim=-1)), dim=-1)            logits = w[:, 0:1] * l_mean + w[:, 1:2] * l_max + w[:, 2:3] * l_first            return logits, self.critic(p_mean).squeeze(-1)    class LoopedTransformer(nn.Module):        def __init__(self, cfg):            super().__init__()            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)            self.pos_emb = nn.Embedding(cfg.max_formula_len, cfg.d_model)            self.blocks = nn.ModuleList([                nn.ModuleList([                    RMSNorm(cfg.d_model),                    QKNormAttention(cfg.d_model, cfg.nhead),                    RMSNorm(cfg.d_model),                    SwiGLU(cfg.d_model, cfg.dim_feedforward),                ])                for _ in range(cfg.num_layers)            ])            self.final_norm = RMSNorm(cfg.d_model)            self.mtp_head = MTPHead(cfg.d_model, cfg.vocab_size)            self.num_loops = cfg.num_loops        def forward(self, x, mask=None):            pos = torch.arange(x.shape[1], device=x.device).unsqueeze(0)            h = self.tok_emb(x) + self.pos_emb(pos)            for _ in range(self.num_loops):                for norm1, attn, norm2, ffn in self.blocks:                    h = h + attn(norm1(h), mask)                    h = h + ffn(norm2(h))            logits, value = self.mtp_head(self.final_norm(h))            return logits.unsqueeze(1), value    return LoopedTransformer(config).to(config.device)# ============================================================# 9. 核心訓練器: GRPOAlphaTrainer (v5.9 多樣性修復)# ============================================================class StableRankMonitor:    def __init__(self):        self.history = []    def compute(self, model):        import torch        ranks = []        with torch.no_grad():            for name, param in model.named_parameters():                if "weight" in name and param.dim() >= 2:                    W = param.data.float()                    frobenius_sq = (W ** 2).sum()                    v = torch.randn(W.shape[1], device=W.device)                    for _ in range(10):                        u = W @ v                        u = u / (u.norm() + 1e-8)                        v = W.T @ u                        v = v / (v.norm() + 1e-8)                    spectral_norm = (W @ v).norm()                    if spectral_norm > 1e-8:                        ranks.append((frobenius_sq / (spectral_norm ** 2)).item())        if ranks:            return {"avg_rank": sum(ranks) / len(ranks)}        return {}def _cosine_similarity(a: List[int], b: List[int]) -> float:    """計算兩個公式 token 序列的餘弦相似度"""    if not a or not b or len(a) != len(b):        return 0.0    a_arr = np.array(a, dtype=np.float32)    b_arr = np.array(b, dtype=np.float32)    dot = np.dot(a_arr, b_arr)    norm_a = np.linalg.norm(a_arr)    norm_b = np.linalg.norm(b_arr)    if norm_a < 1e-8 or norm_b < 1e-8:        return 0.0    return float(dot / (norm_a * norm_b))class GRPOAlphaTrainer:    def __init__(self, config=None, gh_pusher=None):        self.config = config or GRPOConfig.auto_detect()        self.vm = StackVM()        self.reward_calc = GRPORewardCalculator(self.config)        self.model, self.optimizer = None, None        self.history = []        self.gh_pusher = gh_pusher    def init_torch(self):        import torch        # 【v6.17 fix】禁用 torch dynamo 避免 PyTorch 2.10+ functorch grad_and_value 錯誤        torch._dynamo.config.suppress_errors = True        try:            torch._dynamo.disable()        except:            pass        self.model = build_looped_transformer(self.config)        self.optimizer = torch.optim.Adam(            self.model.parameters(), lr=self.config.lr, weight_decay=1e-5        )        self.scaler = torch.amp.GradScaler(            "cuda", enabled=(self.config.device == "cuda")        )    def _make_reseed_formula(self, feature_weights=None, min_len=4, max_attempts=5):        """v6.19: Re-seed 公式生成 — 品質過濾 + 長度符合 min_formula_len"""        for attempt in range(max_attempts):            tf = (np.argsort(-np.array(feature_weights))[:N_FEATURES] if feature_weights is not None                  else np.arange(N_FEATURES))            n_features_in = np.random.randint(2, min(4, N_FEATURES))            n_ops_in = min_len - n_features_in            if n_ops_in <= 0:                n_ops_in = 1                n_features_in = min_len - 1            tokens = [int(tf[i % N_FEATURES]) for i in range(n_features_in)]            valid_ops = [N_FEATURES + i for i in range(N_OPERATORS)]            tokens += [int(np.random.choice(valid_ops)) for _ in range(n_ops_in)]            if hasattr(self, 'feat_tensor') and self.feat_tensor is not None:                signal = self.vm.execute(tokens, self.feat_tensor)                if signal is not None and np.std(signal) > 1e-4:                    ic = self._compute_ic(signal, self.returns)                    if ic > -0.02:                        return tokens            else:                return tokens        return [0, 1, N_FEATURES, N_FEATURES + 1]    def _get_temperature(self, step: int) -> float:        """v5.9: Temperature scheduling — high探索 → low利用"""        if step >= self.config.temperature_decay_steps:            return self.config.temperature_end        progress = step / self.config.temperature_decay_steps        t = self.config.temperature_start * (1 - progress) + self.config.temperature_end * progress        return t    def _apply_top_k_nucleus(self, logits, k=0, threshold=0.0):        """v5.9: Top-k + Nucleus sampling 過濾"""        import torch        if k > 0:            top_k = min(k, logits.size(-1))            indices_to_remove = logits < torch.topk(logits, top_k)[0][:, -1].unsqueeze(-1)            logits = logits.masked_fill(indices_to_remove, -1e9)        if threshold > 0.0 and threshold < 1.0:            sorted_logits, sorted_indices = torch.sort(logits, descending=True)            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)            sorted_indices_to_remove = cumulative_probs > threshold            sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()            sorted_indices_to_remove[:, 0] = False            indices_to_remove = sorted_indices_to_remove.scatter(                1, sorted_indices, sorted_indices_to_remove            )            logits = logits.masked_fill(indices_to_remove, -1e9)        return logits    def _compute_ic_array(self, group_tokens, feat_tensor=None, returns=None):        if feat_tensor is None or returns is None:            return np.zeros(len(group_tokens))        ics = []        for tokens in group_tokens:            signal = self.vm.execute(tokens, feat_tensor)            if signal is None or np.std(signal) < 1e-4:                ics.append(0.0)                continue            valid = np.isfinite(signal) & np.isfinite(returns)            if valid.sum() < 10:                ics.append(0.0)                continue            ic = GRPORewardCalculator._spearman_corr(signal[valid], returns[valid])            ics.append(ic if not np.isnan(ic) else 0.0)        return np.array(ics)    def train_torch_regime(self, feat_tensor, returns, regime_plan=None,                           val_feat=None, val_returns=None):        import torch        if self.model is None:            self.init_torch()                feature_mask = regime_plan.get("feature_mask") if regime_plan else None        operator_mask = regime_plan.get("operator_mask") if regime_plan else None        feature_weights = regime_plan.get("feature_weights") if regime_plan else None        regime_name = (regime_plan["regime"].value                       if regime_plan and hasattr(regime_plan.get("regime"), "value")                       else "unknown")                if regime_plan and "group_size" in regime_plan:            self.config.group_size = regime_plan["group_size"]                # 【v6.13】GPU/CPU 都設最低 group_size = 16        self.config.group_size = max(self.config.group_size, 16)                if feature_mask is not None and operator_mask is not None:            mask = np.ones(VOCAB_SIZE, dtype=np.float32) * -1e9            for i in range(N_FEATURES):                if feature_mask[i]:                    mask[i] = 0.0            for i in range(N_OPERATORS):                if operator_mask[i]:                    mask[N_FEATURES + i] = 0.0            self._precomputed_regime_mask = torch.tensor(mask, device=self.config.device)                if feature_weights is not None:            fw = torch.tensor(feature_weights, device=self.config.device)            fw_logits = torch.zeros(VOCAB_SIZE, device=self.config.device)            fw_logits[:N_FEATURES] = torch.log(fw + 0.01)            self._precomputed_fw_logits = fw_logits * 0.5                best_formula, best_reward, best_composite, history = None, -float("inf"), -float("inf"), []        G = self.config.group_size        # 【v6.11】Early stopping 追蹤        best_val_ic = -float("inf")        patience_counter = 0        best_step = 0                print(f"  [GRPO] regime={regime_name}, G={G}, device={self.config.device}, "              f"steps={self.config.train_steps}, lr={self.config.lr}")                # v6.11: Dynamic Group Size 追蹤        current_group_size = self.config.group_size        collapse_count = 0        for step in range(self.config.train_steps):            self.model.train()            all_log_probs, all_tokens, all_entropies = [], [], []            is_warmup = step < self.config.warmup_steps            temperature = self._get_temperature(step)                        # 【v6.15A】計算 operator seed 數量            n_operator_seeds = int(G * self.config.operator_seed_ratio) if is_warmup else 0            for g in range(G):                # 【v6.15A】Warmup: operator_seed_ratio 的 sample 用預填 operator 公式模板                if is_warmup and g < n_operator_seeds:                    top_feats = (np.argsort(-np.array(feature_weights))[:N_FEATURES]                                 if feature_weights is not None else np.arange(N_FEATURES))                    if g % 3 == 0:                        # 二元 operator: RPN f1 f2 ADD                        binary_ops = [N_FEATURES, N_FEATURES+1, N_FEATURES+2]  # ADD, SUB, MUL                        op = binary_ops[(g // 3) % len(binary_ops)]                        f1_idx = int(top_feats[(g // 3) % N_FEATURES])                        f2_idx = int(top_feats[((g // 3) + 1) % N_FEATURES])                        seed_tokens = [f1_idx, f2_idx, op]                    elif g % 3 == 1:                        # 一元 operator: RPN f NEG                        unary_ops = [N_FEATURES+4, N_FEATURES+5, N_FEATURES+6]  # NEG, ABS, SIGN                        op = unary_ops[(g // 3) % len(unary_ops)]                        f_idx = int(top_feats[(g // 2) % N_FEATURES])                        seed_tokens = [f_idx, op]                    else:                        # 二元變體: RPN f1 f2 MUL                        f1_idx = int(top_feats[(g // 2) % N_FEATURES])                        f2_idx = int(top_feats[((g // 2) + 2) % N_FEATURES])                        seed_tokens = [f1_idx, f2_idx, N_FEATURES+2]  # MUL                    # 【v6.15A】計算 model 對 seed_tokens 的 log_prob（autoregressive）                    seed_log_probs = []                    seed_entropies = []                    for si, st in enumerate(seed_tokens):                        if si == 0:                            seed_inp = torch.zeros(1, 1, dtype=torch.long, device=self.config.device)                        else:                            seed_inp_t = [0] + seed_tokens[:si]                            seed_inp = torch.tensor([seed_inp_t], dtype=torch.long, device=self.config.device)                        seed_lgt, _ = self.model(seed_inp)                        lgt = seed_lgt[:, -1, :].squeeze(0)                        dist = torch.distributions.Categorical(logits=_safe_logits(lgt))                        action = torch.tensor(st, device=self.config.device)                        seed_log_probs.append(dist.log_prob(action))                        seed_entropies.append(dist.entropy())                    all_tokens.append(seed_tokens)                    all_log_probs.append(torch.stack(seed_log_probs).sum())                    all_entropies.append(torch.stack(seed_entropies).mean())                    continue                                vm_state = StackVMState(max_stack_depth=self.config.max_stack_depth)                inp = torch.zeros(1, 1, dtype=torch.long, device=self.config.device)                token_list, log_probs, entropies = [], [], []                # 【v6.15C】計算當前 epsilon（每 500 步衰減）                current_epsilon = (self.config.operator_epsilon_start *                                   self.config.operator_epsilon_decay ** (step // 500))                # 【v6.15D】計算當前 feature logit bias（線性歸零）                if step < self.config.feature_logit_bias_decay_steps:                    feat_logit_bias = (self.config.feature_logit_bias_start *                                       (1.0 - step / self.config.feature_logit_bias_decay_steps))                else:                    feat_logit_bias = 0.0                                for t_pos in range(self.config.max_formula_len):                    remaining = self.config.max_formula_len - t_pos                    if vm_state.is_complete() and len(token_list) >= 1:                        break                    valid_tokens = (vm_state.get_valid_tokens(t_pos, remaining)                                    if self.config.guided_decoding else None)                    if self.config.guided_decoding and not valid_tokens:                        break                                        logits, _ = self.model(inp)                    logits_last = logits[:, -1, :].squeeze(0).clone()                                        # Guided decoding mask                    if valid_tokens is not None:                        guided_mask = torch.full((VOCAB_SIZE,), -1e9, device=self.config.device)                        guided_mask[list(valid_tokens)] = 0.0                        logits_last = logits_last + guided_mask                                        # Regime mask + feature weights                    if hasattr(self, '_precomputed_regime_mask'):                        logits_last = logits_last + self._precomputed_regime_mask                    if hasattr(self, '_precomputed_fw_logits'):                        logits_last = logits_last + self._precomputed_fw_logits                                        # 【v6.15D】Feature logit bias — position 0 壓制 feature logits                    if t_pos == 0 and feat_logit_bias != 0.0:                        feature_bias = torch.zeros(VOCAB_SIZE, device=self.config.device)                        feature_bias[:N_FEATURES] = feat_logit_bias                        logits_last = logits_last + feature_bias                                        # v5.9: Gumbel noise injection (softmax 前)                    if self.config.gumbel_noise_scale > 0:                        gumbel_noise = -torch.log(                            -torch.log(torch.rand_like(logits_last) + 1e-10) + 1e-10                        )                        logits_last = logits_last + gumbel_noise * self.config.gumbel_noise_scale / temperature                                        # v5.9: Temperature scaling                    logits_last = logits_last / temperature                                        # v5.9: Top-k + Nucleus sampling                    logits_last = self._apply_top_k_nucleus(                        logits_last.unsqueeze(0),                        k=self.config.top_k,                        threshold=self.config.nucleus_threshold,                    ).squeeze(0)                                        dist = torch.distributions.Categorical(logits=_safe_logits(logits_last))                    action = dist.sample()                                        # 【v6.15C】Epsilon-greedy operator injection — position 0 時強制選 operator                    # 【v6.16 Fix】跳過 guided mask！P0 bug: guided decoding 阻擋 operator                    if t_pos == 0 and vm_state.stack_depth == 0 and current_epsilon > 0.01:                        if np.random.random() < current_epsilon:                            # 直接從全部 operator 選取，跳過 guided mask                            forced_op = int(np.random.choice(range(N_FEATURES, VOCAB_SIZE)))                            action = torch.tensor(forced_op, dtype=torch.long, device=self.config.device)                            log_probs.append(dist.log_prob(action))                            entropies.append(dist.entropy())                            token_list.append(action.item())                            vm_state.apply_token(action.item())                            inp = torch.cat([inp, action.view(1, 1)], dim=1)                            continue                    log_probs.append(dist.log_prob(action))                    entropies.append(dist.entropy())                    token_list.append(action.item())                    vm_state.apply_token(action.item())                    inp = torch.cat([inp, action.view(1, 1)], dim=1)                                # Fallback: 不完整公式 → 用最強特徵                if not vm_state.is_complete() or len(token_list) < 1:                    feat_idx = (int(np.argmax(feature_weights))                                if feature_weights is not None else 0)                    fb_inp = torch.zeros(1, 1, dtype=torch.long, device=self.config.device)                    fb_logits, _ = self.model(fb_inp)                    fb_dist = torch.distributions.Categorical(                        logits=_safe_logits(fb_logits[:, -1, :].squeeze(0))                    )                    fb_action = torch.tensor(feat_idx, device=self.config.device)                    token_list = [feat_idx]                    log_probs = [fb_dist.log_prob(fb_action)]                    entropies = [fb_dist.entropy()]                                all_tokens.append(token_list)                all_log_probs.append(torch.stack(log_probs).sum())                all_entropies.append(                    torch.stack(entropies).sum() if entropies                    else torch.tensor(0.0, device=self.config.device)                )                        # 計算 IC            train_ic = self._compute_ic_array(all_tokens, feat_tensor, returns)            val_ic = (self._compute_ic_array(all_tokens, val_feat, val_returns)                      if val_feat is not None else np.zeros(len(all_tokens)))                        # 計算 reward            result = self.reward_calc.compute_group_rewards(                all_tokens, feat_tensor, returns, train_ic=train_ic, val_ic=val_ic            )                        # v5.9: Diversity reward — 同 group 公式相似度懲罰            diversity_bonus = 0.0            if self.config.diversity_penalty > 0:                for i in range(len(all_tokens)):                    for j in range(i + 1, len(all_tokens)):                        sim = _cosine_similarity(all_tokens[i], all_tokens[j])                        if sim > 0.9:                            diversity_bonus -= self.config.diversity_penalty * (sim - 0.9)                # 加入 diversity bonus 到 rewards                diversity_per = diversity_bonus / max(len(all_tokens), 1)                        advantages = torch.tensor(result["advantages"], dtype=torch.float32, device=self.config.device)            advantages = torch.nan_to_num(advantages, nan=0.0)                        # v6.11: adv_std 由 compute_group_rewards 計算（Rank-Based 保證 std≈0.577）            adv_std = result.get("adv_std", advantages.std().item())            # v6.11: 不再需要噪聲注入！Rank-Based advantage 永不 collapse            # v6.11: Dynamic Group Size — 如果 need_reduce_group，下步縮小 group            if result.get("need_reduce_group", False) and current_group_size > self.config.min_group_size:                current_group_size = max(self.config.min_group_size, current_group_size // 2)                collapse_count += 1                print(f"  [v6.11] Reducing group_size → {current_group_size} (collapse #{collapse_count})")            elif not result.get("need_reduce_group", True) and current_group_size < self.config.group_size:                current_group_size = min(self.config.group_size, current_group_size * 2)            log_probs_tensor = torch.stack([lp.reshape(()) for lp in all_log_probs])            ratio = torch.exp(log_probs_tensor - log_probs_tensor.detach())            clipped_ratio = torch.clamp(ratio, 1.0 - self.config.clip_eps,                                        1.0 + self.config.clip_eps)            loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()                        # Entropy bonus            if all_entropies:                loss -= self.config.entropy_coef * torch.stack(all_entropies).mean()                        # Diversity bonus            if diversity_bonus != 0.0:                loss += diversity_per * 0.1                        self.optimizer.zero_grad()            if not (torch.isnan(loss) or torch.isinf(loss)):                self.scaler.scale(loss).backward()                self.scaler.unscale_(self.optimizer)                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)                self.scaler.step(self.optimizer)                self.scaler.update()                        if any(p.isnan().any() for p in self.model.parameters()):                print(f"  [v6.5] NaN detected in params, reinitializing model")                self.init_torch()                        # 【v6.19 P0】Composite score 全遍歷選擇 — 取代 reward-argmax            composites = []            for j in range(len(all_tokens)):                if len(all_tokens[j]) < self.config.min_formula_len:                    composites.append(-999.0)                    continue                j_tic = float(train_ic[j]) if train_ic is not None and j < len(train_ic) else 0.0                j_vic = max(float(val_ic[j]), 0.0) if val_ic is not None and j < len(val_ic) else 0.0                j_ops = sum(1 for t in all_tokens[j] if t >= N_FEATURES)                c = (j_tic * self.config.composite_ic_weight +                     j_vic * self.config.composite_val_ic_weight +                     self.config.composite_operator_tiny_bonus * (j_ops > 0))                composites.append(c)            best_composite_idx = int(np.argmax(composites))            if composites[best_composite_idx] > best_composite:                best_composite = composites[best_composite_idx]                best_reward = result["rewards"][best_composite_idx]                best_formula = all_tokens[best_composite_idx]                best_step = step                patience_counter = 0                        # 【v6.12】Early stopping: 修正 val_IC 取值 + 增加 warmup            # 【v6.16 Fix】僅在 best_ops>0 且 best_len>2 時允許早停            if (hasattr(self.config, "early_stop_patience")                and step % 200 == 0 and step > 0                and step > self.config.early_stop_warmup):                current_val_ic = float(np.max(val_ic)) if val_ic is not None and len(val_ic) > 0 else best_reward * 0.1                # 【v6.19】Early stop has_exploration 引用 composite-best                best_composite_toks = all_tokens[best_composite_idx] if best_composite_idx < len(all_tokens) else []                best_composite_n_ops = sum(1 for t in best_composite_toks if t >= N_FEATURES)                has_exploration = best_composite_n_ops > 0 and len(best_composite_toks) > 2                if current_val_ic > best_val_ic + self.config.early_stop_min_delta:                    best_val_ic = current_val_ic                    patience_counter = 0                else:                    if has_exploration:                        patience_counter += 200                    else:                        patience_counter = max(0, patience_counter - 100)                if patience_counter >= self.config.early_stop_patience and has_exploration:                    print(f"  [v6.16] Early stopping at step {step}: val_IC 停滯 {patience_counter} steps (best_val_ic={best_val_ic:.4f})")                    break                        # 【v6.16】探索狀態計算            n_with_ops_t = sum(1 for t in all_tokens if any(tok >= N_FEATURES for tok in t))            avg_ops_t = np.mean([sum(1 for tok in t if tok >= N_FEATURES) for t in all_tokens])            avg_len_t = np.mean([len(t) for t in all_tokens])            # 【v6.16】Periodic Operator Re-seed            if (step % self.config.periodic_reseed_interval == 0 and step > 0                and step >= self.config.warmup_steps):                reseed_n = int(G * self.config.periodic_reseed_ratio)                reseed_idx = np.random.choice(G, min(reseed_n, G), replace=False)                for rg in reseed_idx:                    all_tokens[rg] = self._make_reseed_formula(feature_weights, min_len=self.config.min_formula_len)            # 【v6.18】強制重種觸發降溫 — best_composite 連續 1000 步未更新            # 【v6.19】best_composite 停滯觸發強制降溫重種            if (step > self.config.exploration_stagnation_first_step                and step - best_step >= self.config.exploration_stagnation_steps                and best_composite <= self.config.exploration_stagnation_composite_threshold):                print(f"  [v6.19] best_composite 鎖定 {step - best_step} 步 (val={best_composite:.4f}) → 強制探索重啟")                current_epsilon = self.config.exploration_stagnation_eps_target                feat_logit_bias = self.config.feature_logit_bias_start                reseed_n = int(G * self.config.exploration_stagnation_reseed_ratio)                reseed_idx = np.random.choice(G, min(reseed_n, G), replace=False)                for rg in reseed_idx:                    all_tokens[rg] = self._make_reseed_formula(feature_weights, min_len=self.config.min_formula_len)            # 【v6.16】Closed-Loop: 探索崩潰檢測 + 反向增強            if (self.config.adaptive_exploration and step >= self.config.warmup_steps                and step % 200 == 0                and (avg_ops_t < self.config.exploration_recovery_ops_threshold or avg_len_t < self.config.exploration_recovery_len_threshold)):                current_epsilon = min(current_epsilon * 1.8, self.config.exploration_recovery_eps_boost)                feat_logit_bias = min(feat_logit_bias - 0.5, self.config.exploration_recovery_fbias_boost)                reseed_n = int(G * self.config.exploration_recovery_reseed_ratio)                reseed_idx = np.random.choice(G, min(reseed_n, G), replace=False)                for rg in reseed_idx:                    all_tokens[rg] = self._make_reseed_formula(feature_weights, min_len=self.config.min_formula_len)            if step % 200 == 0:                # 【v6.18 P0】best_val_ic 獨立追蹤 — 不限 early stop warmup                if val_ic is not None and len(val_ic) > 0:                    _current_val_ic = float(np.max(val_ic))                    if _current_val_ic > best_val_ic:                        best_val_ic = _current_val_ic                # 【v6.15】統計 operator exploration                n_with_ops = n_with_ops_t                avg_ops = avg_ops_t                avg_len = avg_len_t                print(f"  step {step:5d}: loss={loss.item():.4f} "                      f"mean_r={result['group_mean_reward']:.3f} "                      f"best_r={best_reward:.3f} temp={temperature:.2f} "                      f"adv_std={adv_std:.4f} method={self.config.advantage_method} ")                print(f"               [v6.15] eps={current_epsilon:.3f} fbias={feat_logit_bias:.2f} "                      f"with_ops={n_with_ops}/{len(all_tokens)} avg_ops={avg_ops:.2f} avg_len={avg_len:.1f}")                best_toks = all_tokens[best_composite_idx] if best_composite_idx < len(all_tokens) else []                best_n_ops = sum(1 for t in best_toks if t >= N_FEATURES)                # 【v6.13】顯示 reward 分項 debug                best_detail = result.get("reward_details", [{}])                if result["best_idx"] < len(best_detail):                    det = best_detail[result["best_idx"]]                    if isinstance(det, dict) and not det.get("_invalid", False):                        print(f"               best_ops={best_n_ops} best_len={len(best_toks)} val_ic_best={best_val_ic:.4f} best_composite={best_composite:.4f}")                        print(f"               [v6.13] ic={det.get('ic_comp',0):.3f} cplx={det.get('complexity_comp',0):.3f} simp={det.get('simplicity_comp',0):.3f} len_b={det.get('length_comp',0):.3f}")                    else:                        print(f"               best_ops={best_n_ops} best_len={len(best_toks)} val_ic_best={best_val_ic:.4f} best_composite={best_composite:.4f} [invalid]")                else:                    print(f"               best_ops={best_n_ops} best_len={len(best_toks)} val_ic_best={best_val_ic:.4f} best_composite={best_composite:.4f}")                # 【v6.19】JSONL 結構化 Logging        if step % 200 == 0:            log_entry = {                "step": int(step),                "best_composite": round(float(best_composite), 6),                "best_composite_idx": int(best_composite_idx),                "best_reward": round(float(best_reward), 6),                "val_ic_best": round(float(best_val_ic), 6),                "with_ops": int(n_with_ops),                "avg_ops": round(float(avg_ops), 4),                "avg_len": round(float(avg_len), 2),                "temperature": round(float(temperature), 4),                "eps": round(float(current_epsilon), 4),                "fbias": round(float(feat_logit_bias), 4),                "best_formula_len": len(best_formula) if best_formula else 0,                "best_ops": int(best_composite_n_ops),            }            print(f"[LOG] {json.dumps(log_entry)}")        return {            "best_formula": best_formula,            "best_reward": best_reward,            "regime": regime_name,            "n_steps": step + 1,  # 【v6.12】實際訓練步數            "best_step": best_step,  # 【v6.12】最佳公式的 step            "best_val_ic": best_val_ic,  # 【v6.12】最佳 val_IC            "best_composite": best_composite,  # 【v6.18】最佳 composite score            "history": history,        }    def train_all_regimes(self, stock_data_map: Dict[str, dict]):        results = {}        regime_groups = defaultdict(list)                for stock_id, data in stock_data_map.items():            regime = data.get("regime_plan", {}).get("regime", StockRegime.MID_CAP_TECH)            regime_key = regime.value if hasattr(regime, "value") else str(regime)            regime_groups[regime_key].append(stock_id)                print(f"\n[Multi-Regime] 分群結果:")        for rk, stocks in regime_groups.items():            print(f"  {rk}: {stocks} ({len(stocks)} 檔)")                # 【v6.18】regimes_to_train 過濾        if self.config.regimes_to_train:            original_keys = list(regime_groups.keys())            for rk in original_keys:                if rk not in self.config.regimes_to_train:                    print(f"  [v6.18] 跳過 regime={rk} (不在 regimes_to_train 中)")                    del regime_groups[rk]            print(f"  [v6.18] 訓練 regimes: {list(regime_groups.keys())}")        for regime_key, stocks in regime_groups.items():            print(f"\n{'=' * 60}")            print(f"  訓練 regime={regime_key} ({len(stocks)} 檔)")            print(f"{'=' * 60}")                        all_train_feat, all_train_returns = [], []            all_val_feat, all_val_returns = [], []            regime_plan = None                        for stock_id in stocks:                data = stock_data_map[stock_id]                feat, ret = data.get("feat"), data.get("returns")                if feat is not None and ret is not None:                    n_train = int(ret.shape[0] * 0.8)                    all_train_feat.append(feat[:, :n_train])                    all_train_returns.append(ret[:n_train])                    all_val_feat.append(feat[:, n_train:])                    all_val_returns.append(ret[n_train:])                    if regime_plan is None:                        regime_plan = data.get("regime_plan")                        if not all_train_feat:                print(f"  [SKIP] regime={regime_key}: 無有效數據")                continue                        # v5.9 關鍵: 橫斷面拼接 — 同 regime 所有股票的數據合併            train_feat = np.concatenate(all_train_feat, axis=1)            train_returns = np.concatenate(all_train_returns, axis=0)            val_feat = np.concatenate(all_val_feat, axis=1)            val_returns = np.concatenate(all_val_returns, axis=0)                        print(f"  [數據] train: {train_feat.shape}, val: {val_feat.shape}")            print(f"  [數據] train_returns: {train_returns.shape}, "                  f"std={train_returns.std():.4f}, mean={train_returns.mean():.6f}")                        # 每個 regime 重新初始化模型            self.model, self.optimizer = None, None            self.init_torch()                        result = self.train_torch_regime(                train_feat, train_returns, regime_plan, val_feat, val_returns            )                        val_ic = self._compute_ic_array([result["best_formula"]], val_feat, val_returns)[0]            train_ic = self._compute_ic_array([result["best_formula"]], train_feat, train_returns)[0]                        for stock_id in stocks:                results[stock_id] = {                    "best_formula": result["best_formula"],                    "best_reward": result["best_reward"],                    "regime": regime_key,                    "train_ic": train_ic,                    "val_ic": val_ic,                    "ic_gap": train_ic - val_ic,                    "n_steps": result["n_steps"],                    "decoded_formula": decode_formula(result["best_formula"]),                }                return results# ============================================================# 10. Utils / Validation / Main# ============================================================def decode_formula(tokens: List[int]) -> str:    if not tokens:        return "INVALID"    stack = []    for t in tokens:        if t < N_FEATURES:            stack.append(FEATURE_NAMES[t])        else:            op_idx = t - N_FEATURES            if op_idx >= N_OPERATORS:                return "INVALID"            arity = OPERATOR_ARITY[op_idx]            if len(stack) < arity:                return "INVALID"            if arity == 1:                stack.append(f"{OPERATOR_NAMES[op_idx]}({stack.pop()})")            elif arity == 2:                b, a = stack.pop(), stack.pop()                stack.append(f"({a} {OPERATOR_NAMES[op_idx]} {b})")            elif arity == 3:                c, b, a = stack.pop(), stack.pop(), stack.pop()                stack.append(f"GATE({a},{b}|{c})")    return stack[0] if len(stack) == 1 else "INVALID"def main():    check_environment()        # 嘗試載入真實數據, fallback 合成數據    data_path = "/kaggle/input"        # 載入真實數據 (自動掃描 /kaggle/input/ 下所有 CSV)    print(f"\n--- 步驟 1: 載入真實數據 ({data_path}) ---")    result = adapt_finmind_data(data_path)    if result[0] is None:        print("  [FATAL] 找不到真實數據！請確認 Kaggle dataset 已正確掛載。")        print("  [FATAL] 預期路徑存在？", os.path.exists(data_path))        import sys; sys.exit(1)    df, inst_df, margin_df, futures_oi_df, us_indices_df = result        print(f"  載入完成: {len(df)} 筆, {df['stock_id'].nunique()} 檔股票")        # --- 步驟 2: 特徵工程 ---    print("\n--- 步驟 2: 特徵工程 ---")    feat_df = TWFeatureEngineer.compute_features(        df, inst_df, margin_df, futures_oi_df, us_indices_df    )        stock_data_map = {}    planner = RegimeTrainingPlan()    regime_count = defaultdict(int)        for stock_id, group in feat_df.groupby("stock_id"):        regime = KNOWN_REGIMES.get(stock_id, StockRegime.MID_CAP_TECH)        regime_count[regime.value] += 1        feat_cols = [            group[f].values if f in group.columns else np.zeros(len(group))            for f in FEATURE_NAMES        ]        feat_tensor = np.nan_to_num(            np.array(feat_cols, dtype=np.float32), nan=0.0, posinf=5.0, neginf=-5.0        )                horizon = planner.create_plan(stock_id, regime)["reward_horizon"]        close = group["close"].values        fwd_returns = np.zeros(len(close), dtype=np.float32)        for i in range(len(close) - horizon):            fwd_returns[i] = (close[i + horizon] - close[i]) / (close[i] + 1e-6)                stock_data_map[stock_id] = {            "feat": feat_tensor,            "returns": fwd_returns,            "regime_plan": planner.create_plan(stock_id, regime),        }        print(f"\n  Regime 分佈:")    for rk, cnt in sorted(regime_count.items()):        print(f"    {rk}: {cnt} 檔")        # --- 步驟 3: 訓練 ---    print("\n--- 步驟 3: GRPO 訓練 ---")    # 【v6.1】GPU 不可用時只訓練 MID_CAP_TECH regime    try:        import torch        _gpu_avail = torch.cuda.is_available()        _force_cpu = os.environ.get("GRPO_FORCE_CPU", "0") == "1"        if _force_cpu or not _gpu_avail:            print("\n  [v6.12] CPU 模式 → 訓練所有 4 個 regime (保留 regime-specific group_size)")            print(f"  [v6.19] 股票數: {len(stock_data_map)} 檔 (只保留 MID_CAP_TECH)")            # 【v6.19】CPU 模式只保留 MID_CAP_TECH regime 的股票            stock_data_map = {k: v for k, v in stock_data_map.items()                              if v.get("regime") == StockRegime.MID_CAP_TECH.value}            # v6.12 Fix: CPU 模式保留 regime-specific group_size，由 regime_plan 動態覆蓋        else:            print(f"\n  [v6.12] GPU 模式 → 訓練所有 regime, {len(stock_data_map)} 檔")    except ImportError:        print("  [v6.11] 無法 import torch, 使用完整訓練")    trainer = GRPOAlphaTrainer(GRPOConfig.auto_detect())    results = trainer.train_all_regimes(stock_data_map)        print("\n" + "=" * 60 + "\n  訓練完成!\n" + "=" * 60)    final_formulas = {}    for stock_id, res in results.items():        regime = res['regime']        if regime not in final_formulas:            final_formulas[regime] = res        for regime, res in final_formulas.items():        print(f"  Regime [{regime}]: "              f"train_IC={res.get('train_ic', 0):.4f} "              f"val_IC={res.get('val_ic', 0):.4f} "              f"→ {res.get('decoded_formula', 'N/A')}")        # 輸出 best_strategy_per_regime.json    regime_strategies = {}    for regime, res in final_formulas.items():        regime_strategies[regime] = {            "formula_tokens": res.get("best_formula", []),            "formula_str": res.get("decoded_formula", "N/A"),            "train_ic": float(res.get("train_ic", 0)),            "val_ic": float(res.get("val_ic", 0)),            "best_reward": float(res.get("best_reward", 0)),            "n_steps": res.get("n_steps", 0),        }        output = {        "regimes": regime_strategies,        "metadata": {            "vocab_size": VOCAB_SIZE,            "n_features": N_FEATURES,            "all_feature_names": list(FEATURE_NAMES),            "version": "v6.19",            "n_stocks": len(stock_data_map),        },    }        with open("best_strategy_per_regime.json", "w") as f:        json.dump(output, f, indent=2, ensure_ascii=False)    print(f"\n  輸出: best_strategy_per_regime.json")        # 同時輸出 training_report    report = {        "version": "v6.19",        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),        "config": {            "device": trainer.config.device,            "group_size": trainer.config.group_size,            "train_steps": trainer.config.train_steps,            "lr": trainer.config.lr,            "entropy_coef": trainer.config.entropy_coef,            "gumbel_noise_scale": trainer.config.gumbel_noise_scale,            "diversity_penalty": trainer.config.diversity_penalty,            "temperature_range": f"{trainer.config.temperature_start}→{trainer.config.temperature_end}",            "operator_bonus": trainer.config.operator_bonus,            "short_formula_penalty": trainer.config.short_formula_penalty,            "min_formula_len": trainer.config.min_formula_len,            "early_stop_patience": trainer.config.early_stop_patience,            "early_stop_warmup": trainer.config.early_stop_warmup,        },        "regime_results": {            regime: {                "formula": res.get("decoded_formula", "N/A"),                "train_ic": float(res.get("train_ic", 0)),                "val_ic": float(res.get("val_ic", 0)),                "ic_gap": float(res.get("ic_gap", 0)),                "best_reward": float(res.get("best_reward", 0)),            }            for regime, res in final_formulas.items()        },    }    with open("training_report.json", "w") as f:        json.dump(report, f, indent=2, ensure_ascii=False)    print(f"  輸出: training_report.json")if __name__ == "__main__":    main()